# Module 01: Trigger a Flow via HTTP

## Learning Objectives
By completing this notebook, you will:
1. Set up an HTTP-triggered cloud flow in Power Automate
2. Send an HTTP POST request from Python to trigger that flow
3. Pass structured JSON data from Python into the flow
4. Parse the HTTP response returned by the flow
5. Understand how to use HTTP triggers for pipeline integration

## Prerequisites
- Completed Guide 01: Creating Your First Cloud Flow
- An active Microsoft 365 work or school account
- Python 3.8+ with the `requests` library (`pip install requests`)

## Estimated Time: 15 minutes

---

## Why HTTP Triggers?

The scheduled flow in Guide 01 runs on a timer — useful for daily reports, but not for triggering automation in response to events in your own code. An **HTTP trigger** (also called the "When an HTTP request is received" trigger) turns your flow into a REST API endpoint. You can call it from:

- Python scripts and notebooks
- CI/CD pipelines (GitHub Actions, Azure DevOps)
- Other applications and services
- Shell scripts via `curl`

This pattern is fundamental for integrating Power Automate with data engineering pipelines.

## Part 1: Setting Up the HTTP-Triggered Flow in Power Automate

Before writing Python code, create the flow that will receive the HTTP request. Follow these steps in Power Automate.

### Step 1.1 — Create an Instant Cloud Flow

> **On screen:** Go to [https://make.powerautomate.com](https://make.powerautomate.com). In the left rail click **+ Create**. Under "Start from blank", click **Instant cloud flow**.

The "Build an instant cloud flow" dialog opens.

> **On screen:** In the **Flow name** field type `HTTP Triggered Flow — Module 01`. In the trigger list, scroll down and select **When an HTTP request is received**. Click **Create**.

### Step 1.2 — Configure the HTTP Trigger

> **On screen:** The flow opens with a single card: **When an HTTP request is received**. Click the card to expand it.

You will see a field labelled **Request Body JSON Schema**. This schema tells Power Automate what JSON structure to expect in the request body, and it generates typed dynamic content tokens from those fields.

> **On screen:** Click **Use sample payload to generate schema**. Paste the following JSON and click **Done**:

```json
{
    "message": "Hello from Python",
    "priority": "high",
    "timestamp": "2026-03-08T08:00:00Z"
}
```

Power Automate generates a JSON schema from this sample and displays it in the field. The dynamic content panel for downstream steps will now offer `message`, `priority`, and `timestamp` as typed tokens.

### Step 1.3 — Add a Compose Action (to echo the input)

> **On screen:** Click **+ New step**. Search for `Compose`. Select the **Compose** action (under Data Operation).

> **On screen:** Click inside the **Inputs** field of the Compose card. Click **Add dynamic content**. From the panel, under the **When an HTTP request is received** group, click **message**. The token `[message]` appears in the Inputs field.

### Step 1.4 — Add a Response Action

> **On screen:** Click **+ New step**. Search for `Response`. Select the **Response** action (under Request).

> **On screen:** Configure the Response card:
- **Status Code:** `200`
- **Headers:** Click **Add a header**. Key: `Content-Type`, Value: `application/json`
- **Body:** Type the following, inserting dynamic tokens where shown:

```json
{
    "status": "received",
    "echo_message": "[message]",
    "echo_priority": "[priority]",
    "flow_processed_at": "@{utcNow()}"
}
```

For the `@{utcNow()}` expression: click inside the Body field, position the cursor after `"flow_processed_at": "`, then click **Add dynamic content**, switch to the **Expression** tab, type `utcNow()`, and click **OK**.

### Step 1.5 — Save and Copy the Endpoint URL

> **On screen:** Click **Save** in the toolbar.

> **On screen:** After saving, expand the **When an HTTP request is received** card. An **HTTP POST URL** field appears with a long URL. Click the copy icon next to it.

This URL is the endpoint you will call from Python. It looks like:

```
https://prod-XX.westeurope.logic.azure.com:443/workflows/XXXXXXXX.../triggers/manual/paths/invoke?api-version=2016-06-01&sp=%2Ftriggers%2Fmanual%2Frun&sv=1.0&sig=XXXXXXXX
```

The URL contains an embedded SAS (Shared Access Signature) token. Anyone with this URL can trigger your flow. Treat it like a password — do not commit it to version control.

## Part 2: Triggering the Flow from Python

With the flow saved and the endpoint URL copied, we can trigger it using the `requests` library.

In [ ]:
# Purpose: Import dependencies and define the flow endpoint
# The requests library sends HTTP requests; json formats the body payload.

import requests
import json
import os
from datetime import datetime, timezone

# Paste your flow's HTTP POST URL here.
# Best practice: load from an environment variable rather than hardcoding.
# In your terminal: export FLOW_ENDPOINT_URL="https://prod-XX..."

FLOW_ENDPOINT_URL = os.environ.get(
    "FLOW_ENDPOINT_URL",
    "PASTE_YOUR_FLOW_URL_HERE"  # replace this when running the notebook
)

print(f"Endpoint configured: {'Yes' if 'logic.azure.com' in FLOW_ENDPOINT_URL else 'No — paste your URL above'}")

### 2.1 — Basic POST Request

An HTTP-triggered Power Automate flow expects:
- Method: `POST`
- Header: `Content-Type: application/json`
- Body: JSON matching the schema you defined on the trigger card

The `requests.post()` call handles all of this cleanly.

In [ ]:
# Purpose: Send a single HTTP POST request to trigger the flow
# Why requests.post with json=: it sets Content-Type automatically and serialises the dict.

def trigger_flow(endpoint_url: str, payload: dict, timeout_seconds: int = 30) -> requests.Response:
    """
    Trigger a Power Automate HTTP-triggered flow.

    Parameters
    ----------
    endpoint_url : str
        The HTTP POST URL from the flow's trigger card.
    payload : dict
        JSON-serialisable dict matching the flow's request body schema.
    timeout_seconds : int
        Maximum seconds to wait for the flow to respond (default 30).
        Power Automate flows with a Response action reply synchronously;
        flows without one return 202 Accepted immediately.

    Returns
    -------
    requests.Response
        The HTTP response from the flow.

    Raises
    ------
    requests.exceptions.Timeout
        If the flow does not respond within timeout_seconds.
    requests.exceptions.ConnectionError
        If the endpoint URL is unreachable.
    """
    response = requests.post(
        url=endpoint_url,
        json=payload,          # sets Content-Type: application/json and serialises the dict
        timeout=timeout_seconds
    )
    return response


# Build the payload to send
payload = {
    "message": "Hello from Python",
    "priority": "high",
    "timestamp": datetime.now(timezone.utc).isoformat()
}

print("Payload to send:")
print(json.dumps(payload, indent=2))

In [ ]:
# Purpose: Execute the request and display the raw response
# Replace FLOW_ENDPOINT_URL with your actual URL before running.

if "PASTE_YOUR_FLOW_URL_HERE" in FLOW_ENDPOINT_URL:
    print("Skipping request — paste your flow URL in the cell above first.")
else:
    response = trigger_flow(FLOW_ENDPOINT_URL, payload)

    print(f"HTTP Status:  {response.status_code}")
    print(f"Status text:  {response.reason}")
    print(f"Response body:\n{response.text}")

### 2.2 — Parsing the Response

The flow's **Response** action returns a JSON body. We parse it with `response.json()` and access each field directly.

Expected response structure (from the Response action we configured):

```json
{
    "status": "received",
    "echo_message": "Hello from Python",
    "echo_priority": "high",
    "flow_processed_at": "2026-03-08T08:00:00.0000000Z"
}
```

In [ ]:
# Purpose: Parse the JSON response and extract individual fields
# Why check status_code first: a non-200 response body may not be valid JSON.

def parse_flow_response(response: requests.Response) -> dict:
    """
    Parse and validate the JSON response from an HTTP-triggered flow.

    Parameters
    ----------
    response : requests.Response
        The response object from trigger_flow().

    Returns
    -------
    dict
        Parsed response body.

    Raises
    ------
    ValueError
        If the HTTP status code indicates failure or the body is not JSON.
    """
    if response.status_code == 202:
        # 202 Accepted: the flow was triggered but has no Response action.
        # It is running asynchronously. Check Run History in the portal.
        return {"status": "accepted", "note": "Flow triggered asynchronously (no Response action)"}

    if response.status_code != 200:
        raise ValueError(
            f"Flow returned HTTP {response.status_code}: {response.text}\n"
            f"Check the flow's Run History in the Power Automate portal for details."
        )

    # 200 OK: the Response action ran successfully
    try:
        return response.json()
    except requests.exceptions.JSONDecodeError as exc:
        raise ValueError(
            f"Response body is not valid JSON: {response.text}"
        ) from exc


# Run the parse (only if we got a real response above)
if "PASTE_YOUR_FLOW_URL_HERE" not in FLOW_ENDPOINT_URL:
    try:
        result = parse_flow_response(response)
        print("Parsed response:")
        print(json.dumps(result, indent=2))

        # Access individual fields
        print(f"\nFlow status:         {result.get('status')}")
        print(f"Message echoed back: {result.get('echo_message')}")
        print(f"Priority received:   {result.get('echo_priority')}")
        print(f"Flow processed at:   {result.get('flow_processed_at')}")

    except ValueError as err:
        print(f"Error: {err}")
else:
    print("Skipping parse — no response available yet.")

## Part 3: Passing Structured Data

The real power of HTTP triggers is passing arbitrary structured data from Python into the flow. The flow can route, transform, or store that data using its connector library.

### 3.1 — Richer Payload

Update the flow's JSON schema to accept a richer payload:

> **On screen:** In Power Automate, expand the **When an HTTP request is received** trigger card. Click **Use sample payload to generate schema** again and paste:

```json
{
    "report_title": "Daily Sales Summary",
    "author": "data-pipeline",
    "record_count": 1547,
    "status": "complete",
    "tags": ["sales", "daily", "q1-2026"]
}
```

> **On screen:** Click **Done**, then **Save** the flow.

In [ ]:
# Purpose: Build and send a richer structured payload
# This demonstrates how a data pipeline would notify a flow about a completed job.

def build_pipeline_notification(
    report_title: str,
    record_count: int,
    status: str,
    tags: list[str]
) -> dict:
    """
    Construct a payload representing a data pipeline completion event.

    Parameters
    ----------
    report_title : str
        Human-readable name of the completed report.
    record_count : int
        Number of records processed.
    status : str
        Outcome: 'complete', 'partial', or 'failed'.
    tags : list[str]
        Labels for routing or filtering in the flow.

    Returns
    -------
    dict
        JSON-serialisable payload.
    """
    return {
        "report_title": report_title,
        "author": "data-pipeline",
        "record_count": record_count,
        "status": status,
        "tags": tags,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }


# Build a realistic pipeline notification payload
notification_payload = build_pipeline_notification(
    report_title="Daily Sales Summary",
    record_count=1547,
    status="complete",
    tags=["sales", "daily", "q1-2026"]
)

print("Notification payload:")
print(json.dumps(notification_payload, indent=2))

In [ ]:
# Purpose: Send the richer payload and confirm receipt
# Same trigger_flow() function works for any payload shape.

if "PASTE_YOUR_FLOW_URL_HERE" not in FLOW_ENDPOINT_URL:
    notification_response = trigger_flow(FLOW_ENDPOINT_URL, notification_payload)
    print(f"HTTP Status: {notification_response.status_code}")

    if notification_response.status_code == 200:
        print("Flow acknowledged the notification:")
        print(json.dumps(notification_response.json(), indent=2))
    elif notification_response.status_code == 202:
        print("Flow accepted the notification (running asynchronously).")
    else:
        print(f"Unexpected response: {notification_response.text}")
else:
    print("Skipping send — paste your flow URL in Part 2 first.")

## Part 4: Understanding HTTP Response Codes

Power Automate HTTP triggers return specific status codes depending on how the flow is configured and whether it succeeded.

| Status Code | Meaning | When it occurs |
|-------------|---------|----------------|
| **200 OK** | Flow completed and Response action returned 200 | Flow has a Response action, all steps succeeded |
| **202 Accepted** | Flow was queued, no Response action present | Flow has no Response action (async pattern) |
| **400 Bad Request** | Request body does not match the expected schema | Sent wrong field names or wrong data types |
| **404 Not Found** | The flow URL is invalid or the flow was deleted | URL was changed or the flow was removed |
| **500 Internal Server Error** | A step inside the flow failed | Check Run History in the portal |

### Synchronous vs. Asynchronous Triggers

- **With a Response action:** The HTTP POST call blocks until the flow completes (or times out at 120 seconds). You get a 200 with the response body.
- **Without a Response action:** The HTTP POST call returns 202 immediately. The flow runs in the background. Check Run History to see if it succeeded.

In [ ]:
# Purpose: Robust trigger function that handles all response codes
# Production code should always handle the async (202) case explicitly.

def trigger_flow_robust(endpoint_url: str, payload: dict, timeout_seconds: int = 30) -> dict:
    """
    Trigger a flow and return a structured result regardless of response code.

    Parameters
    ----------
    endpoint_url : str
        The HTTP POST URL from the flow's trigger card.
    payload : dict
        Request body payload.
    timeout_seconds : int
        Request timeout in seconds.

    Returns
    -------
    dict
        Keys: 'success' (bool), 'status_code' (int), 'data' (dict or None), 'error' (str or None)
    """
    try:
        response = requests.post(url=endpoint_url, json=payload, timeout=timeout_seconds)
    except requests.exceptions.Timeout:
        return {
            "success": False,
            "status_code": None,
            "data": None,
            "error": f"Request timed out after {timeout_seconds}s. "
                     "Check if the flow has a Response action and that all steps complete within 120s."
        }
    except requests.exceptions.ConnectionError as exc:
        return {
            "success": False,
            "status_code": None,
            "data": None,
            "error": f"Connection failed: {exc}. Verify the endpoint URL is correct."
        }

    if response.status_code == 200:
        # Synchronous success — flow completed and returned a response body
        try:
            data = response.json()
        except requests.exceptions.JSONDecodeError:
            data = {"raw": response.text}
        return {"success": True, "status_code": 200, "data": data, "error": None}

    if response.status_code == 202:
        # Asynchronous acceptance — flow is running in the background
        return {
            "success": True,
            "status_code": 202,
            "data": {"note": "Flow accepted and running asynchronously. Check Run History for result."},
            "error": None
        }

    # All other codes indicate a problem
    return {
        "success": False,
        "status_code": response.status_code,
        "data": None,
        "error": f"HTTP {response.status_code}: {response.text[:500]}"  # cap at 500 chars
    }


# Demonstrate the robust function
if "PASTE_YOUR_FLOW_URL_HERE" not in FLOW_ENDPOINT_URL:
    result = trigger_flow_robust(FLOW_ENDPOINT_URL, payload)
    print(f"Success:      {result['success']}")
    print(f"Status code:  {result['status_code']}")
    if result['data']:
        print(f"Data:         {json.dumps(result['data'], indent=2)}")
    if result['error']:
        print(f"Error:        {result['error']}")
else:
    # Show the function structure even without a live endpoint
    demo_result = trigger_flow_robust.__doc__
    print("trigger_flow_robust() is ready. Paste your flow URL to test it.")

## Exercises

Apply what you have learned by completing these exercises. Each exercise has auto-tests that check your implementation.

### Exercise 4.1: Build a Payload Validator

**Task:** Write a function that validates a payload dict before sending it to the flow. The function should ensure the payload contains all required fields and that their types are correct.

**Required fields and types:**
- `message` — `str`, non-empty
- `priority` — `str`, one of: `"low"`, `"medium"`, `"high"`
- `timestamp` — `str`, non-empty (format validation not required)

**Returns:** `(bool, str)` — `(True, "")` if valid, `(False, "error description")` if invalid.

**Hints:**
- Use `isinstance()` to check types
- Check for empty strings with `len(value) == 0`
- Use an `in` check for the allowed priority values

In [ ]:
# YOUR CODE HERE
# ---------------

def validate_flow_payload(payload: dict) -> tuple[bool, str]:
    """
    Validate a flow trigger payload before sending.

    Parameters
    ----------
    payload : dict
        The payload dict to validate.

    Returns
    -------
    tuple[bool, str]
        (True, "") if the payload is valid.
        (False, "reason") if it is invalid.
    """
    # Replace `pass` with your implementation
    pass

In [ ]:
# AUTO-TESTS — do not modify
# --------------------------

def test_exercise_4_1():
    # Valid payload passes
    valid, msg = validate_flow_payload({"message": "hello", "priority": "high", "timestamp": "2026-01-01T00:00:00Z"})
    assert valid is True, f"Expected True for valid payload, got False with message: '{msg}'"
    assert msg == "", f"Expected empty error string for valid payload, got: '{msg}'"

    # Missing 'message' key fails
    valid, msg = validate_flow_payload({"priority": "low", "timestamp": "2026-01-01T00:00:00Z"})
    assert valid is False, "Expected False when 'message' field is missing"
    assert len(msg) > 0, "Error message should explain what is missing"

    # Empty message string fails
    valid, msg = validate_flow_payload({"message": "", "priority": "high", "timestamp": "2026-01-01T00:00:00Z"})
    assert valid is False, "Expected False for empty 'message' string"

    # Invalid priority value fails
    valid, msg = validate_flow_payload({"message": "test", "priority": "urgent", "timestamp": "2026-01-01T00:00:00Z"})
    assert valid is False, "Expected False for priority='urgent' (not in allowed values)"

    # Wrong type for priority fails
    valid, msg = validate_flow_payload({"message": "test", "priority": 1, "timestamp": "2026-01-01T00:00:00Z"})
    assert valid is False, "Expected False when priority is an int instead of a str"

    # Missing timestamp fails
    valid, msg = validate_flow_payload({"message": "test", "priority": "low"})
    assert valid is False, "Expected False when 'timestamp' field is missing"

    print("Exercise 4.1 passed — validate_flow_payload works correctly.")

test_exercise_4_1()

### Exercise 4.2: Retry Logic

**Task:** Write a function `trigger_with_retry()` that calls `trigger_flow_robust()` and retries up to `max_retries` times if the response indicates a transient failure (status code 500 or a `ConnectionError`). Add a fixed delay between retries.

**Signature:**
```python
def trigger_with_retry(
    endpoint_url: str,
    payload: dict,
    max_retries: int = 3,
    retry_delay_seconds: float = 2.0
) -> dict:
```

**Returns:** The result dict from `trigger_flow_robust()` — either the first success or the last failure after all retries are exhausted.

**Hints:**
- Use a `for` loop over `range(max_retries + 1)` — first attempt plus retries
- Check `result['success']` to decide whether to retry
- Use `time.sleep(retry_delay_seconds)` between attempts
- Return immediately on success — do not wait for remaining retries

In [ ]:
import time

# YOUR CODE HERE
# ---------------

def trigger_with_retry(
    endpoint_url: str,
    payload: dict,
    max_retries: int = 3,
    retry_delay_seconds: float = 2.0
) -> dict:
    """
    Trigger a flow with automatic retry on transient failures.

    Parameters
    ----------
    endpoint_url : str
        The HTTP POST URL from the flow's trigger card.
    payload : dict
        Request body payload.
    max_retries : int
        Number of additional attempts after the first (default 3).
    retry_delay_seconds : float
        Seconds to wait between attempts (default 2.0).

    Returns
    -------
    dict
        Result dict from trigger_flow_robust().
    """
    # Replace `pass` with your implementation
    pass

In [ ]:
# AUTO-TESTS — do not modify
# --------------------------

def test_exercise_4_2():
    # Use a mock to control trigger_flow_robust behaviour without a real endpoint
    call_count = [0]
    results_to_return = [
        {"success": False, "status_code": 500, "data": None, "error": "Server error"},
        {"success": False, "status_code": 500, "data": None, "error": "Server error"},
        {"success": True,  "status_code": 200, "data": {"status": "received"}, "error": None},
    ]

    def mock_trigger(url, payload, timeout_seconds=30):
        idx = call_count[0]
        call_count[0] += 1
        return results_to_return[idx]

    # Temporarily replace trigger_flow_robust with the mock
    import builtins
    original = globals().get('trigger_flow_robust')
    globals()['trigger_flow_robust'] = mock_trigger

    try:
        result = trigger_with_retry("http://fake-url", {"message": "test", "priority": "low", "timestamp": "t"}, max_retries=3, retry_delay_seconds=0)
        assert result['success'] is True, f"Expected success after retries, got: {result}"
        assert call_count[0] == 3, f"Expected 3 calls (2 failures + 1 success), got {call_count[0]}"
    finally:
        globals()['trigger_flow_robust'] = original

    # Test that it returns the last failure when all retries are exhausted
    call_count[0] = 0
    all_fail = [{"success": False, "status_code": 500, "data": None, "error": "fail"}] * 10

    def mock_always_fail(url, payload, timeout_seconds=30):
        return {"success": False, "status_code": 500, "data": None, "error": "persistent failure"}

    globals()['trigger_flow_robust'] = mock_always_fail
    try:
        result = trigger_with_retry("http://fake-url", {}, max_retries=2, retry_delay_seconds=0)
        assert result['success'] is False, "Expected failure when all retries exhausted"
    finally:
        globals()['trigger_flow_robust'] = original

    print("Exercise 4.2 passed — trigger_with_retry handles retries correctly.")

test_exercise_4_2()

## Summary

### Key Takeaways

1. **HTTP triggers** make any Power Automate flow callable as a REST API from Python, shell scripts, or any HTTP client.
2. **The endpoint URL embeds authentication** (SAS token) — treat it like a password and load it from environment variables, not hardcoded strings.
3. **Flows with a Response action** respond synchronously (200 OK with a JSON body). **Flows without one** respond asynchronously (202 Accepted).
4. **Structured JSON payloads** defined via the "Use sample payload" dialog become typed dynamic content tokens accessible to every action in the flow.
5. **Production callers** should validate payloads before sending and implement retry logic for transient failures.

### What Is Next

- **Guide 02:** Testing and Debugging — reading run history, inspecting inputs/outputs, diagnosing errors
- **Module 02:** Triggers and Connectors — the full trigger catalog: SharePoint, Teams, Dataverse, and scheduled triggers with advanced recurrence patterns
- **Exercise 01:** `exercises/01_first_flow_exercise.py` — construct and validate a Power Automate flow definition JSON from scratch

### Additional Resources

- [Power Automate — When an HTTP request is received trigger](https://learn.microsoft.com/en-us/azure/connectors/connectors-native-reqres)
- [Power Automate — Response action](https://learn.microsoft.com/en-us/azure/connectors/connectors-native-reqres#add-a-response-action)
- [requests library documentation](https://requests.readthedocs.io/en/latest/)